## Threading

In [ ]:
import time
import threading

In [ ]:
threading.current_thread()

Without threads

In [ ]:
import time

def task(n):
    time.sleep(1)
    print(n)

start = time.time()
for i in range(3):
    task(i)

print(time.time() - start)

With threads

In [ ]:
threads = []
start = time.time()
for i in range(3):
    t = threading.Thread(target=task, args=(i,))
    t.start()
    threads.append(t)

for t in threads:
    t.join()

print(time.time() - start)

## Daemon thread

In [ ]:
import threading, time

def worker():
    while True:
        print("working")
        time.sleep(1)

threading.Thread(target=worker, daemon=True).start()
print("main finished")

## Lock. Rlock

### Демонстрация работы Lock

In [ ]:
import time
import threading
from threading import Lock

delay = 0.0001
counter = 0
lock = Lock()

def increment_eternal():
    while True:
        global counter
        counter += 1
        print(counter)

def execute():
    global counter
    tmp = counter
    time.sleep(delay)
    counter = tmp + 1
    print(counter)
        
def increment_artificial(enable_lock=False):
    if enable_lock:
        with lock:
            execute()
    else:
        execute()
    # lock.acquire()
    # tmp = counter
    # time.sleep(delay)
    # counter = tmp + 1
    # print(counter)
    # lock.release()

threads = []
for _ in range(10):
    t = threading.Thread(target=increment_artificial, args=(False, ))
    t.start()
    threads.append(t)

for t in threads:
    t.join()

print("counter =", counter) 

### Сравнение Lock vs RLock

In [ ]:
from threading import Lock, RLock, Thread
import time

lock = RLock()

def thread_a():
    print("A: acquire")
    lock.acquire()
    print("A: inside critical section")
    time.sleep(2)
    print("A: still thinks lock is held")
    time.sleep(2)
    print("A: release")
    lock.release()

def thread_b():
    time.sleep(1)
    print("B: release чужого lock!")
    lock.release() # обычный Lock НЕ ПРОВЕРЯЕТ, кто владелец

t1 = Thread(target=thread_a)
t2 = Thread(target=thread_b)

t1.start()
t2.start()
t1.join()
t2.join()

### Self-deadlock

In [ ]:
lock = RLock()

def f():
    with lock:
        g()

def g():
    with lock:
        pass
f()

## Semaphore

In [ ]:
import threading
import time
import random
from threading import Semaphore

N = 2
sem = threading.Semaphore(value=N)

in_section = 0
in_section_lock = threading.Lock()

def worker(i: int) -> None:
    global in_section

    print(f"[{i}] wants to enter")
    sem.acquire()
    try:
        with in_section_lock:
            in_section += 1
            current = in_section
        print(f"[{i}] ENTER (in_section={current})")

        # "работа" внутри секции
        time.sleep(random.uniform(0.6, 1.4))

    finally:
        with in_section_lock:
            in_section -= 1
            current = in_section
        sem.release()
        print(f"[{i}] EXIT  (in_section={current})")

threads = [threading.Thread(target=worker, args=(i,)) for i in range(1, 9)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print("Done")


## ThreadPoolExecutor

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def task(n):
    time.sleep(1)
    return n

max_workers = 3
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(task, n) for n in range(3)]
    results = [f.result() for f in futures]

print(results)


In [ ]:
futures[0]